In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, Wedge, Circle, Rectangle
from matplotlib.colors import LinearSegmentedColormap
# Clemson palette
ORANGE = "#F56600"
PURPLE = "#522D80"
GRAY = "#C8C8C8"

sns.set_theme(style="white", font_scale=1.2)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200

### Data Availability

The underlying Fanatics purchase dataset was provided for the application
project and is not included in this public repository. The analysis and
visualizations below demonstrate the methodology used without exposing the
underlying customer-level data.

In [ ]:
#Read in CSV
# The original dataset is not included in this repository.
# The analysis was completed using a private application dataset.
# df = pd.read_csv("FanaticsData.csv")
# Drop duplicate rows
df = df.drop_duplicates()
df["ClickDate"] = pd.to_datetime(df["ClickDate"])
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

In [ ]:
#Question 1: Which email campaign(s) brought in the most revenue?
# Revenue is only populated when a click led to an order, so summing it by campaign gives revenue attributed to each campaign.
campaign_rev = (df.groupby("CampaignName")["Revenue"].sum().sort_values(ascending=False).reset_index())
campaign_rev["Pct"] = campaign_rev["Revenue"] / campaign_rev["Revenue"].sum() * 100
top = campaign_rev.iloc[0]
print(f"Top campaign: {top['CampaignName']} with ${top['Revenue']:,.0f} "
      f"({top['Pct']:.1f}% of total revenue)")
campaign_rev

In [ ]:
#Graph 1
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]

# Clemson palette on white
ORANGE, PURPLE = "#F56600", "#522D80"
MUTEDBAR = "#BDB6CF"                    # lavender-gray for the rest
INK, MUTED = "#1F1A2E", "#6B6480"
CARD, CARD_EDGE, GRID = "#F6F3FB", "#E3DDF0", "#E9E5F1"
n = len(campaign_rev)
W, H = 165, 93
fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1]); ov.set_xlim(0, W); ov.set_ylim(H, 0); ov.axis("off")
def rbox(x, y, w, h, fc, r, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x + r, y + r), w - 2*r, h - 2*r,boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

# LEFT PANEL 
L, LW = 6, 54
rbox(L, 7.2, 1.2, 3.2, ORANGE, 0.5)
ov.text(L + 3, 8.8, "CLEMSON ATHLETICS  ·  FAN COMMERCE", color=MUTED, fontsize=9.5, va="center")
ov.text(L, 18, f"{top['CampaignName']} Drives", color=INK, fontsize=26, fontweight="bold", va="center")
ov.text(L, 25.5, f"{top['Pct']:.0f}% of Email Revenue", color=ORANGE, fontsize=26, fontweight="bold", va="center")
ov.text(L, 33, "Attributed merchandise revenue from 7-day click window", color=MUTED, fontsize=10.5, va="center")

card_h, gap, y0 = 13, 2.5, 39.5
stats = [("TOP CAMPAIGN", top["CampaignName"]),("REVENUE FROM TOP CAMPAIGN", f"${top['Revenue']:,.0f}"),("SHARE OF TOTAL REVENUE", f"{top['Pct']:.1f}%")]
for i, (lab, val) in enumerate(stats):
    y = y0 + i * (card_h + gap)
    rbox(L, y, LW, card_h, CARD, 1.8, ec=CARD_EDGE)
    rbox(L + 1.8, y + 2.4, 1.0, card_h - 4.8, ORANGE, 0.5)
    ov.text(L + 5, y + 4.3, lab, color=MUTED, fontsize=8.5, va="center")
    ov.text(L + 5, y + 9.3, val, color=INK, fontsize=24, fontweight="bold", va="center")
    if i == 2:
        cx, cy, r, th = L + LW - 9.5, y + card_h / 2, 5.2, 1.5
        ov.add_patch(Wedge((cx, cy), r, 0, 360, width=th, fc=CARD_EDGE, ec="none"))
        ov.add_patch(Wedge((cx, cy), r, -90, -90 + 3.6 * top["Pct"], width=th, fc=ORANGE, ec="none"))

ov.plot([L + LW + 5, L + LW + 5], [8, 88], color=GRID, lw=1.2)

# ---------------- RIGHT PANEL: real chart with axes ----------------
RX = 66
ov.text(RX, 9, "Revenue by Email Campaign", color=INK, fontsize=15, fontweight="bold", va="center")
ov.text(RX, 13.8, "Attributed revenue ($) from orders within 7 days of an email click", color=MUTED, fontsize=10.5, va="center")

ax = fig.add_axes([0.505, 0.115, 0.46, 0.70], facecolor="white")
vals = campaign_rev["Revenue"].values
colors = [ORANGE] + [PURPLE] * 2 + [MUTEDBAR] * (n - 3)
y = np.arange(n)

# soft glow behind the winner
for h, a in ((1.15, 0.06), (1.0, 0.10)):
    ax.barh(0, vals[0], height=h, color=ORANGE, alpha=a, zorder=1, lw=0)
bars = ax.barh(y, vals, height=0.68, color=colors, zorder=3, lw=0)
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in vals], padding=6, fontsize=10.5, color=INK)

ax.set_yticks(y)
ax.set_yticklabels(campaign_rev["CampaignName"], fontsize=11)
for i, lab in enumerate(ax.get_yticklabels()):
    lab.set_color(INK if i < 3 else MUTED)
    lab.set_fontweight("bold" if i < 3 else "normal")
ax.invert_yaxis()
ax.set_ylim(n - 0.4, -0.6)

ax.set_xlim(0, vals.max() * 1.17)
ax.set_xticks(np.arange(0, vals.max() + 1, 10000))
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"${v/1000:.0f}K" if v else "$0"))
ax.tick_params(axis="x", labelsize=10, colors=MUTED, length=0, pad=8)
ax.tick_params(axis="y", length=0, pad=10)
ax.xaxis.grid(True, color=GRID, lw=1); ax.set_axisbelow(True)
for s in ("top", "right", "bottom"):
    ax.spines[s].set_visible(False)
ax.spines["left"].set_color(INK); ax.spines["left"].set_linewidth(1.3)   # zero baseline
ov.text(L, H - 3.2, "Notes: Revenue attributed to orders placed within 7 days of an email click. "
        "Orders touched by two campaigns count toward both.", color=MUTED, fontsize=10.5, va="center")
plt.savefig("q1_campaign_revenue.png", dpi=200, facecolor="white")
plt.show()

In [ ]:
#Question 2: What percentage of customers who purchased used a promo code at checkout?
# Customers who made a purchase (a row with Revenue = an order)
purchasers = df[df["Revenue"].notna()]

# Did each purchasing customer use a promo code at least once?
used_promo = purchasers.groupby("CustomerID")["OrderPromo"].apply(lambda s: s.notna().any())
pct_promo = used_promo.mean() * 100
print(f"{used_promo.sum():,} of {len(used_promo):,} purchasing customers used a promo code ({pct_promo:.1f}%)")

In [ ]:
#Graph 2
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]
ORANGE, PURPLE = "#F56600", "#522D80"
MUTEDBAR = "#BDB6CF"
INK, MUTED = "#1F1A2E", "#6B6480"
CARD = "#F6F3FB"

n_promo, n_total = int(used_promo.sum()), len(used_promo)
n_no = n_total - n_promo

W, H = 165, 93
fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1]); ov.set_xlim(0, W); ov.set_ylim(H, 0); ov.axis("off")
ov.set_aspect("equal")
def rbox(x, y, w, h, fc, r, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x + r, y + r), w - 2*r, h - 2*r, boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

# LEFT: purple hero panel
PX, PY, PW, PH = 5, 5, 78, 83
rbox(PX, PY, PW, PH, PURPLE, 3)
rbox(PX + 6, 13, 1.2, 3.2, ORANGE, 0.5)
ov.text(PX + 9, 14.6, "CLEMSON ATHLETICS  ·  FAN COMMERCE", color="#D9CFEA", fontsize=9.5, va="center")
ov.text(PX + 6, 30, f"{pct_promo:.2f}%", color="white", fontsize=52, fontweight="bold", va="center")
ov.text(PX + 6, 44, "of Purchasers used a Promo Code", color=ORANGE, fontsize=21, fontweight="bold", va="center")
ov.text(PX + 6, 50, "at least once at checkout", color="#D9CFEA", fontsize=14, va="center")

# dot strip: 20 dots
k = round(pct_promo / 5)
ov.text(PX + 6, 61, f"ABOUT {k} OF EVERY 20 PURCHASERS", color="#D9CFEA", fontsize=10.5, va="center")
for i in range(20):
    ov.add_patch(Circle((PX + 7.25 + i * 3.45, 67), 1.25, fc=ORANGE if i < k else "#7A5AA6", ec="none"))

# mini stats
for j, (lab, val, col) in enumerate([("USED A PROMO", f"{n_promo:,}", ORANGE), ("NO PROMO", f"{n_no:,}", "#BDB1D6")]):
    x = PX + 6 + j * 34
    ov.text(x, 76, lab, color="#D9CFEA", fontsize=10.5, va="center")
    ov.text(x, 82, val, color=col, fontsize=22, fontweight="bold", va="center")

# RIGHT: donut
RX = 94
ov.text(RX, 11, "Promo Code Usage Among Purchasers", color=INK, fontsize=15, fontweight="bold", va="center")
ov.text(RX, 15.8, "Share of purchasing customers who used a promo code on at least one order", color=MUTED, fontsize=10.5, va="center")

cx, cy, r, th = 125, 52, 27, 8.5
ov.add_patch(Wedge((cx, cy), r + 3.2, 0, 360, width=r + 3.2, fc=CARD, ec="none"))
ov.add_patch(Wedge((cx, cy), r, 0, 360, width=th, fc=MUTEDBAR, ec="none"))
ov.add_patch(Wedge((cx, cy), r, -90, -90 + 3.6 * pct_promo, width=th, fc=ORANGE, ec="none"))
ov.text(cx, cy - 3, f"{n_promo:,}", color=INK, fontsize=34, fontweight="bold", ha="center", va="center")
ov.text(cx, cy + 4.5, f"of {n_total:,} purchasers", color=MUTED, fontsize=11, ha="center", va="center")

for j, (c, lab) in enumerate([(ORANGE, f"Used promo code  ({pct_promo:.2f}%)"),(MUTEDBAR, f"No promo code  ({100-pct_promo:.2f}%)")]):
    x = 104 + j * 31
    ov.add_patch(Circle((x, 87), 1.2, fc=c, ec="none"))
    ov.text(x + 2.5, 87, lab, color=INK, fontsize=9.5, va="center")

ov.text(5, H - 1.6, "Customer-level: a purchaser counts as a promo user if any of their orders carried a promo code. "
        "Purchaser = customer with at least one attributed order.", color=MUTED, fontsize=10.5, va="center")
plt.savefig("q2_promo_usage.png", dpi=200, facecolor="white")
plt.show()

In [ ]:
# Question 3: Which states brought in the most revenue?
orders = df[df["Revenue"].notna()].copy()
orders["State"] = orders["State"].fillna("Unknown")      # keep missing-state orders so totals tie to Q1

state_rev = (orders.groupby("State").agg(Revenue=("Revenue", "sum"),Orders=("Revenue", "size"),Customers=("CustomerID", "nunique")).sort_values("Revenue", ascending=False).reset_index())
state_rev["Pct"] = state_rev["Revenue"] / state_rev["Revenue"].sum() * 100
state_rev["CumPct"] = state_rev["Pct"].cumsum()
state_rev["RevPerCustomer"] = state_rev["Revenue"] / state_rev["Customers"]

known = state_rev[state_rev["State"] != "Unknown"]
print(f"Total attributed revenue: ${state_rev['Revenue'].sum():,.0f}")
print(f"States with revenue: {len(known)}, Revenue with no state listed: "
      f"${state_rev.loc[state_rev['State']=='Unknown','Revenue'].sum():,.0f}")
print(f"Top 3 states = {known['Pct'].head(3).sum():.1f}% of revenue\n")
print(known.head(15).round(1).to_string(index=False))

In [ ]:
#Find Pop Up Shop Place/Region
top_state = known.iloc[0]["State"]
sc = orders[orders["State"] == top_state].copy()
sc["ZipCode"] = sc["ZipCode"].astype(str).str.zfill(5)
sc["Zip3"] = sc["ZipCode"].str[:3]                         # 3-digit prefix = broad region

def summarize(g):
    out = (g.agg(Revenue=("Revenue", "sum"), Orders=("Revenue", "size"), Customers=("CustomerID", "nunique")).sort_values("Revenue", ascending=False).reset_index())
    out["Pct_of_State"] = out["Revenue"] / out["Revenue"].sum() * 100
    return out

zip3_rev = summarize(sc.groupby("Zip3"))
zip_rev  = summarize(sc.groupby("ZipCode"))
print(f"\n{top_state}: revenue by ZIP prefix")
print(zip3_rev.head(6).round(1).to_string(index=False))
print(f"\n{top_state}: top 10 ZIP codes")
print(zip_rev.head(10).round(1).to_string(index=False))

In [ ]:
#Greenville/Charleston Analysis
sc_all = df[df["State"] == "South Carolina"].copy()
sc_all["Zip3"] = sc_all["ZipCode"].astype(str).str.zfill(5).str[:3]
cmp = sc_all[sc_all["Zip3"].isin(["296", "294"])]
summary = cmp.groupby("Zip3").agg(Clicks=("CustomerID", "size"),Customers=("CustomerID", "nunique"),Orders=("Revenue", "count"),Revenue=("Revenue", "sum"))
summary["ConvPct"] = summary["Orders"] / summary["Clicks"] * 100
summary["RevPerOrder"] = summary["Revenue"] / summary["Orders"]
buyers = cmp[cmp["Revenue"].notna()].groupby(["Zip3", "CustomerID"]).size()
summary["RepeatPct"] = (buyers > 1).groupby("Zip3").mean() * 100
print(summary.round(1))

In [ ]:
#Graph 3 - Slide 1: national view (map + bar)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]
ORANGE, PURPLE = "#F56600", "#522D80"
MUTEDBAR = "#BDB6CF"
INK, MUTED = "#1F1A2E", "#6B6480"
CARD, CARD_EDGE, GRID = "#F6F3FB", "#E3DDF0", "#E9E5F1"
NOREV = "#EFEBF6"

ABBR = {"Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA","Colorado":"CO","Connecticut":"CT","Delaware":"DE","District of Columbia":"DC","Florida":"FL",
        "Georgia":"GA","Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD","Massachusetts":"MA",
 "Michigan":"MI","Minnesota":"MN","Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV","New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
 "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK","Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD",
 "Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT","Virginia":"VA","Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY"}
GRID_POS = {"AK":(0,0),"ME":(0,10),"VT":(1,9),"NH":(1,10),"WA":(2,0),"ID":(2,1),"MT":(2,2),"ND":(2,3),"MN":(2,4),"IL":(2,5),"WI":(2,6),"MI":(2,7),"NY":(2,8),"RI":(2,9),"MA":(2,10), "OR":(3,0),
    "NV":(3,1),"WY":(3,2),"SD":(3,3),"IA":(3,4),"IN":(3,5),"OH":(3,6),"PA":(3,7),"NJ":(3,8),"CT":(3,9), "CA":(4,0),"UT":(4,1),"CO":(4,2),"NE":(4,3),"MO":(4,4),"KY":(4,5),"WV":(4,6),
    "VA":(4,7),"MD":(4,8),"DE":(4,9),"AZ":(5,1),"NM":(5,2),"KS":(5,3),"AR":(5,4),"TN":(5,5),"NC":(5,6),"SC":(5,7),"DC":(5,8),"OK":(6,3),"LA":(6,4),"MS":(6,5),"AL":(6,6),"GA":(6,7),"HI":(7,0),
    "TX":(7,3),"FL":(7,8)}

rev_by_abbr = {ABBR[s]: v for s, v in zip(known["State"], known["Revenue"]) if s in ABBR}
unknown_rev = state_rev.loc[state_rev["State"] == "Unknown", "Revenue"].sum()
total_rev = state_rev["Revenue"].sum()
top3 = known.head(3)
top3_abbr = [ABBR[s] for s in top3["State"]]

W, H = 165, 93
fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1]); ov.set_xlim(0, W); ov.set_ylim(H, 0); ov.axis("off")
def rbox(x, y, w, h, fc, r, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x + r, y + r), w - 2*r, h - 2*r, boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

# ---- header ----
L = 6
rbox(L, 7.2, 1.2, 3.2, ORANGE, 0.5)
ov.text(L + 3, 8.8, "CLEMSON ATHLETICS  ·  FAN COMMERCE", color=MUTED, fontsize=9.5, va="center")
ov.text(L, 17.5, f"{known.iloc[0]['State']} Drives", color=INK, fontsize=26, fontweight="bold", va="center")
ov.text(L, 25, f"{known.iloc[0]['Revenue'] / total_rev * 100:.0f}% of Email Revenue", color=ORANGE, fontsize=26, fontweight="bold", va="center")

# ---- tile-grid map ----
MX, MY, P, T = 6, 31, 6.9, 6.3
for ab, (r, c) in GRID_POS.items():
    x, y = MX + c * P, MY + r * P
    rev = rev_by_abbr.get(ab, 0)
    if ab == top3_abbr[0]:   fc, tc = ORANGE, "white"
    elif ab in top3_abbr:    fc, tc = PURPLE, "white"
    elif rev > 0:            fc, tc = MUTEDBAR, INK
    else:                    fc, tc = NOREV, "#A9A1BC"
    rbox(x, y, T, T, fc, 0.9)
    if ab in top3_abbr:
        ov.text(x + T/2, y + 2.6, ab, color=tc, fontsize=11, fontweight="bold", ha="center", va="center")
        ov.text(x + T/2, y + 4.9, f"${rev/1000:.0f}K", color=tc, fontsize=7.5, ha="center", va="center")
    else:
        ov.text(x + T/2, y + T/2, ab, color=tc, fontsize=9.5, fontweight="bold" if rev > 0 else "normal", ha="center", va="center")

lg = [(ORANGE, f"#1 {top3['State'].iloc[0]}"), (PURPLE, f"#2–3 {top3['State'].iloc[1]}, {top3['State'].iloc[2]}"), (MUTEDBAR, "Other states with revenue"), (NOREV, "No attributed revenue")]
for i, (c, lab) in enumerate(lg):
    ov.add_patch(FancyBboxPatch((MX + P + 0.2, MY + 0.4 + i * 3.4), 2.2, 2.2, boxstyle="round,pad=0,rounding_size=0.5", fc=c, ec=CARD_EDGE if c == NOREV else "none", lw=0.8))
    ov.text(MX + P + 3.4, MY + 1.5 + i * 3.4, lab, color=INK, fontsize=9, va="center")
ov.plot([91, 91], [8, 86], color=GRID, lw=1.2)

# ---- right: revenue by state (top 5 + all other states) ----
RX = 96
ov.text(RX, 9, "Revenue by State", color=INK, fontsize=15, fontweight="bold", va="center")
ov.text(RX, 13.8, "Attributed revenue ($) from orders within 7 days of an email click", color=MUTED, fontsize=10.5, va="center")

t5 = known.head(5).reset_index(drop=True)
oth = known.iloc[5:]
rows = pd.concat([t5[["State", "Revenue", "Pct"]], pd.DataFrame({"State": [f"Other states ({len(oth)})"], "Revenue": [oth["Revenue"].sum()], "Pct": [oth["Revenue"].sum() / total_rev * 100]})], 
                 ignore_index=True)
n = len(rows)
CH_TOP, CH_BOT = 24, 82                      # chart band (same bottom edge as the map)
ax = fig.add_axes([0.682, 1 - CH_BOT / H, 0.188, (CH_BOT - CH_TOP) / H], facecolor="none")
cols = [ORANGE, PURPLE, PURPLE, MUTEDBAR, MUTEDBAR, CARD_EDGE]
yy = np.arange(n)
ax.barh(yy, rows["Revenue"], height=0.62, color=cols, zorder=3, lw=0)
ax.set_yticks(yy); ax.set_yticklabels(rows["State"], fontsize=12)
for i, lab in enumerate(ax.get_yticklabels()):
    lab.set_color(INK if i < 3 else MUTED); lab.set_fontweight("bold" if i < 3 else "normal")
ax.set_ylim(n - 0.5, -0.5); ax.set_xlim(0, rows["Revenue"].max() * 1.03); ax.set_xticks([])
ax.tick_params(axis="y", length=0, pad=8)
for s_ in ("top", "right", "bottom"): ax.spines[s_].set_visible(False)
ax.spines["left"].set_color(INK); ax.spines["left"].set_linewidth(1.3)

COL_REV, COL_PCT = 152, 161
row_h = (CH_BOT - CH_TOP) / n
ov.text(COL_REV, CH_TOP - 2.5, "REVENUE", color=MUTED, fontsize=8, ha="right", va="center")
ov.text(COL_PCT, CH_TOP - 2.5, "SHARE", color=MUTED, fontsize=8, ha="right", va="center")
for i, (v, p) in enumerate(zip(rows["Revenue"], rows["Pct"])):
    yc = CH_TOP + (i + 0.5) * row_h
    ov.text(COL_REV, yc, f"${v/1000:.1f}K", color=INK if i < 3 else MUTED, fontsize=12.5, fontweight="bold" if i < 3 else "normal", ha="right", va="center")
    ov.text(COL_PCT, yc, f"{p:.1f}%", color=INK if i < 3 else MUTED, fontsize=12.5, ha="right", va="center")
# thin divider above the "other states" row
ov.plot([RX, W - 4], [CH_TOP + 5 * row_h, CH_TOP + 5 * row_h], color=GRID, lw=1)

# ---- single footnote ----
ov.text(L, H - 3.2, "Notes: Revenue attributed to orders placed within 7 days of an email click. "
        f"${unknown_rev/1000:.1f}K ({unknown_rev/total_rev*100:.0f}%) of revenue has no state listed and is not shown.", color=MUTED, fontsize=10.5, va="center")
plt.savefig("q3_slide1_national.png", dpi=200, facecolor="white")
plt.show()

In [ ]:
#Graph 3 - Slide 2: South Carolina by region + pop-up shop pick (schematic tile map)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]
ORANGE, PURPLE = "#F56600", "#522D80"
MUTEDBAR = "#BDB6CF"
INK, MUTED = "#1F1A2E", "#6B6480"
CARD, CARD_EDGE, GRID = "#F6F3FB", "#E3DDF0", "#E9E5F1"
LAV, LAVDK = "#D9CFEA", "#BDB1D6"
# ---- region data: SC ZIP codes grouped by their first 3 digits ----
# name: (3-digit prefixes, ZIP label, (row, col) on the schematic grid)
REGIONS = {"Upstate":     (["296"], "296", (0, 0)),"Spartanburg": (["293"], "293", (0, 1)),"Rock Hill":   (["297"], "297", (0, 2)),"Columbia":    (["290", "291", "292"], "290–292", (1, 2)),
    "Florence":    (["295"], "295", (1, 3)),"Aiken":       (["298"], "298", (2, 1)),"Charleston":  (["294"], "294", (2, 3)),"Beaufort":    (["299"], "299", (3, 2)),}
sc_r = df[df["State"] == "South Carolina"].copy()
sc_r["Zip3"] = sc_r["ZipCode"].astype(str).str.zfill(5).str[:3]
sc_total_rev = sc_r["Revenue"].sum()
sc_conv = sc_r["Revenue"].notna().sum() / len(sc_r) * 100

reg = {}
for name, (pre, zl, pos) in REGIONS.items():
    g = sc_r[sc_r["Zip3"].isin(pre)]
    o = int(g["Revenue"].notna().sum())
    reg[name] = dict(Clicks=len(g), Orders=o, Revenue=g["Revenue"].sum(), Customers=g["CustomerID"].nunique(),
                     Conv=o / len(g) * 100, RevPerOrder=g["Revenue"].sum() / o)
ch, up = reg["Charleston"], reg["Upstate"]

W, H = 165, 93
fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1]); ov.set_xlim(0, W); ov.set_ylim(H, 0); ov.axis("off")
def rbox(x, y, w, h, fc, r, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x + r, y + r), w - 2*r, h - 2*r, boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

# ---- header ----
L = 6
rbox(L, 7.2, 1.2, 3.2, ORANGE, 0.5)
ov.text(L + 3, 8.8, "CLEMSON ATHLETICS  ·  FAN COMMERCE", color=MUTED, fontsize=9.5, va="center")
ov.text(L, 17.5, "South Carolina by Region", color=INK, fontsize=26, fontweight="bold", va="center")
ov.text(L, 25, "Pop-Up Pick: Charleston", color=ORANGE, fontsize=26, fontweight="bold", va="center")

# ---- schematic tile map ----
MX, MY, TW, TH, GX, GY = 6, 31, 18.5, 12.2, 1.8, 1.5
for name, (pre, zl, (r, c)) in REGIONS.items():
    x, y = MX + c * (TW + GX), MY + r * (TH + GY)
    d = reg[name]
    if name == "Charleston":     fc, tc, sc_ = ORANGE, "white", "#FFE3CF"
    elif d["Conv"] >= sc_conv:   fc, tc, sc_ = PURPLE, "white", LAV
    else:                        fc, tc, sc_ = MUTEDBAR, INK, MUTED
    rbox(x, y, TW, TH, fc, 1.4)
    ov.text(x + 1.8, y + 2.5, name, color=tc, fontsize=10.5, fontweight="bold", va="center")
    ov.text(x + 1.8, y + 5.0, f"ZIP {zl}", color=sc_, fontsize=8.5, va="center")
    ov.text(x + 1.8, y + 8.1, f"{d['Conv']:.1f}%", color=tc, fontsize=16, fontweight="bold", va="center")
    ov.text(x + 1.8, y + 10.7, f"${d['Revenue']/1000:.1f}K · {d['Orders']} orders", color=sc_, fontsize=8.5, va="center")

# legend in the empty middle-left of the grid
lg = [(ORANGE, "Pop-up pick: Charleston"), (PURPLE, f"Above SC average click → order ({sc_conv:.1f}%)"), (MUTEDBAR, "Below SC average")]
ly = MY + 3 * (TH + GY)
for i, (c, lab) in enumerate(lg):
    ov.add_patch(FancyBboxPatch((MX + 0.2, ly + i * 3.6), 2.2, 2.2, boxstyle="round,pad=0,rounding_size=0.5", fc=c, ec="none"))
    ov.text(MX + 3.6, ly + 1.1 + i * 3.6, lab, color=INK, fontsize=9, va="center")
ov.text(MX + 0.2, ly + 3 * 3.6 + 1.2, "Tile figures: click → order rate, revenue, orders", color=MUTED, fontsize=10, va="center")
ov.plot([91, 91], [8, 86], color=GRID, lw=1.2)

# ---- right: pop-up pick card ----
CX, CY, CW, CH = 96, 8, 65, 78
rbox(CX, CY, CW, CH, PURPLE, 2.5)
rbox(CX + 5, CY + 5.6, 1.0, 3.0, ORANGE, 0.4)
ov.text(CX + 7.5, CY + 7.1, "POP-UP SHOP PICK", color=LAV, fontsize=11, va="center")
ov.text(CX + 5, CY + 14.2, "Charleston Area, SC", color="white", fontsize=24, fontweight="bold", va="center")
ov.text(CX + 5, CY + 20.6, "Higher-converting audience, a fraction of the reach", color=LAV, fontsize=12, va="center")
ov.plot([CX + 5, CX + CW - 5], [CY + 24.8, CY + 24.8], color="#6E4A9E", lw=1)

mets = [("CLICK → ORDER", ch["Conv"], up["Conv"], f"{ch['Conv']:.1f}%", f"vs {up['Conv']:.1f}% Upstate"),("REVENUE / ORDER", ch["RevPerOrder"], up["RevPerOrder"], 
        f"${ch['RevPerOrder']:.0f}", f"vs ${up['RevPerOrder']:.0f} Upstate"),("EMAIL AUDIENCE", ch["Customers"], up["Customers"], f"{ch['Customers']:,}", f"vs {up['Customers']:,} Upstate")]
BX, BW = CX + 41, 18
for k, (lab, a, b, val, sub) in enumerate(mets):
    r0 = CY + 27 + k * 14.5
    ov.text(CX + 5, r0 + 1.0, lab, color=LAV, fontsize=10, va="center")
    ov.text(CX + 5, r0 + 6.8, val, color=ORANGE, fontsize=22, fontweight="bold", va="center")
    ov.text(CX + 5, r0 + 10.8, sub, color=LAVDK, fontsize=10, va="center")
    m = max(a, b)
    for j, (nm, v, col) in enumerate([("Charleston", a, ORANGE), ("Upstate", b, LAVDK)]):
        yb = r0 + 4.4 + j * 4.2
        ov.text(CX + 32, yb + 1.3, nm, color=LAV, fontsize=7.5, va="center")
        rbox(BX, yb, max(v / m * BW, 1.4), 2.6, col, 0.6)
ov.plot([CX + 5, CX + CW - 5], [CY + 71.5, CY + 71.5], color="#6E4A9E", lw=1)
ov.text(CX + 5, CY + 75, f"{ch['Orders']} orders  ·  ${ch['Revenue']/1000:.1f}K revenue  ·  {ch['Revenue']/sc_total_rev*100:.0f}% of SC revenue", color=LAV, fontsize=10.5, va="center")

# ---- footnote ----
ov.text(L, H - 3.2, "Notes: Regions group SC ZIP codes by their first 3 digits (Columbia = 290–292; Upstate = 296; Charleston = 294). "
        , color=MUTED, fontsize=10, va="center")
plt.savefig("q3_slide2_south_carolina.png", dpi=200, facecolor="white")
plt.show()

In [ ]:
# Question 4: Who is our biggest 'superfan'?
orders = df[df["Revenue"].notna()].copy()
# Some orders show up twice (one order credited to two campaigns). Keep one row per order so spend isn't double counted.
unique_orders = orders.drop_duplicates(["CustomerID", "OrderDate", "Revenue"])

spend = (unique_orders.groupby("CustomerID").agg(Revenue=("Revenue", "sum"), Orders=("Revenue", "size"),Promo=("OrderPromo", lambda s: ", ".join(sorted(s.dropna().unique())))))
engage = (df.groupby("CustomerID").agg(Clicks=("ClickDate", "nunique"), Campaigns=("CampaignName", "nunique"),State=("State", "first"), ZipCode=("ZipCode", "first")))
fans = spend.join(engage).sort_values("Revenue", ascending=False).reset_index()

top = fans.iloc[0]
median_spend, avg_spend = fans["Revenue"].median(), fans["Revenue"].mean()
print(f"Biggest superfan: Customer {top['CustomerID']:.0f} | ${top['Revenue']:,.0f} across {top['Orders']:.0f} orders "f"| {top['Clicks']:.0f} email clicks, {top['Campaigns']:.0f} campaigns | {top['State']} {top['ZipCode']}")
print(f"That's {top['Revenue']/median_spend:.1f}x the median purchaser (${median_spend:,.0f}) "f"and {top['Revenue']/avg_spend:.1f}x the average (${avg_spend:,.0f}); {len(fans):,} purchasers total\n")
print(fans.head(10).round(0).to_string(index=False))

In [ ]:
# GRAPH 4 - SLIDE 1: BIGGEST SUPER SPENDER
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]
ORANGE, PURPLE = "#F56600", "#522D80"
INK, MUTED = "#1F1A2E", "#6B6480"
CARD, CARD_EDGE = "#F6F3FB", "#E3DDF0"

# Customer calculations
top = fans.iloc[0]
spender_id = int(top["CustomerID"])
median_spend = fans["Revenue"].median()
uo = unique_orders.copy()
uo["Day"] = uo["OrderDate"].dt.date
days = uo.groupby("CustomerID")["Day"].nunique()
loy = pd.DataFrame({"Days": days, "Rev": uo.groupby("CustomerID")["Revenue"].sum()})
loy = loy.sort_values(["Days", "Rev"], ascending=False)
loyal_id = int(loy.index[0])
loyal_days = int(loy.iloc[0]["Days"])
loyal_rev = loy.iloc[0]["Rev"]
n_multi = int((days >= 2).sum())
n_max = int((days >= loyal_days).sum())

def journey(cid):
    out = []
    for cd, g in df[df["CustomerID"] == cid].groupby("ClickDate"):
        o = g[g["Revenue"].notna()].drop_duplicates(["OrderDate", "Revenue"]).sort_values("OrderDate")
        out.append((cd, list(zip(o["OrderDate"], o["Revenue"])), g["CampaignName"].iloc[0]))
    return sorted(out, key=lambda t: t[0])

J = {spender_id: journey(spender_id), loyal_id: journey(loyal_id)}
MAXO = max(r for j in J.values() for _, os_, _ in j for _, r in os_)

# Design settings
W, H = 165, 93
TX, TW = 5, 155

def canvas():
    fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
    ov = fig.add_axes([0, 0, 1, 1])
    ov.set_xlim(0, W); ov.set_ylim(H, 0); ov.axis("off")
    return fig, ov

def rbox(ov, x, y, w, h, fc, r, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x+r, y+r), w-2*r, h-2*r, boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

def header(ov, title, sub):
    rbox(ov, 6, 7.2, 1.2, 3.2, ORANGE, 0.5)
    ov.text(9, 8.8, "Clemson Athletics    Fan Commerce", color=MUTED, fontsize=11, va="center")
    ov.text(6, 17.5, title, color=INK, fontsize=26, fontweight="bold", va="center")
    ov.text(6, 25, sub, color=ORANGE, fontsize=26, fontweight="bold", va="center")

def footnote(ov, line1):
    ov.text(6, H-4, line1, color=MUTED, fontsize=9.5, va="center")
    ov.text(6, H-1.7, f"Timeline is to scale by date; same-day events are spaced slightly apart for readability. Customer IDs are anonymous. {len(fans):,} customers made a purchase.", color=MUTED, fontsize=9.5, va="center")

def timeline_card(ov, cid, eyebrow, tag, line3, TY=31, TH=38, band=False):
    rbox(ov, TX, TY, TW, TH, CARD, 3, ec=CARD_EDGE, lw=1.2)
    rbox(ov, TX+4, TY+3.2, 1, 3, ORANGE, 0.4)
    ov.text(TX+6.5, TY+4.7, eyebrow, color=MUTED, fontsize=11, va="center")
    ov.add_patch(Circle((62, TY+4.7), 1, fc="white", ec=PURPLE, lw=1.6))
    ov.text(64.2, TY+4.7, "Email Click", color=MUTED, fontsize=10.5, va="center")
    ov.add_patch(Circle((81, TY+4.7), 1.3, fc=ORANGE, ec="none"))
    ov.text(83.5, TY+4.7, "Order (Larger Dot = Larger Order)", color=MUTED, fontsize=10.5, va="center")
    dts = [t for cd, os_, _ in J[cid] for t in [cd] + [d for d, _ in os_]]
    amin, amax = min(dts)-pd.Timedelta(days=15), max(dts)+pd.Timedelta(days=15)
    X0, X1 = 54, 154
    X = lambda t: X0 + (t-amin)/(amax-amin)*(X1-X0)
    y, AXY = TY+17.5, TY+30.5
    if band:
        multi = [(cd, os_, c) for cd, os_, c in J[cid] if len(os_) >= 2]
        if multi:
            cd, os_, camp = multi[0]
            sx = X(pd.Timestamp(os_[0][0]).normalize()+pd.Timedelta(hours=12))
            ov.add_patch(Rectangle((sx-6.2, TY+8), 11.6, AXY-TY-8, fc="#E7DDF6", ec="none", zorder=1.5))
            ov.text(sx+5.4, TY+4.7, f"Same Day  ·  {camp}", color=PURPLE, fontsize=10, fontweight="bold", ha="right", va="center", zorder=2)
    ov.plot([X0, X1], [AXY, AXY], color=INK, lw=1.1)
    months = pd.date_range(amin, amax, freq="MS")
    for m in months:
        ov.plot([X(m), X(m)], [AXY, AXY+1.1], color=INK, lw=1)
        ov.text(X(m), AXY+3, f"{m:%b}", color=MUTED, fontsize=9, ha="center", va="center")
        if m.month in (1, 9) or m == months[0]:
            ov.text(X(m), AXY+5.3, f"{m:%Y}", color=INK, fontsize=9, fontweight="bold", ha="center", va="center")
    ov.text(TX+4, y-3.6, f"Customer {cid}", color=INK, fontsize=13, fontweight="bold", va="center")
    ov.text(TX+4, y, tag, color=MUTED, fontsize=10, va="center")
    ov.text(TX+4, y+3.8, line3, color=ORANGE, fontsize=11, fontweight="bold", va="center")
    ov.plot([X0, X1], [y, y], color="#DDD6EC", lw=1.4, zorder=1)
    prev_click_x, prev_click_t = None, None

    for cd, os_, camp_ in J[cid]:
        x0, n = X(cd), len(os_)
        radii = [1.0] + [0.9+1.25*np.sqrt(r/MAXO) for _, r in os_]
        pos = [0.0]

        for k in range(1, len(radii)):
            pos.append(pos[-1]+radii[k-1]+radii[k]+0.7)

        seq = [x0+p-np.mean(pos) for p in pos]
        ov.add_patch(Circle((seq[0], y), 1, fc="white", ec=PURPLE, lw=1.6, zorder=3))

        for (od, rev), ox, rr in zip(os_, seq[1:], radii[1:]):
            ov.add_patch(Circle((ox, y), rr, fc=ORANGE, ec="white", lw=0.8, zorder=4))

        if n:
            lab = " + ".join(f"\\${r:,.0f}" for _, r in os_)
            ov.text(np.mean(seq[1:]), y-4, lab, color=INK, fontsize=9.5, fontweight="bold", ha="center", va="center", zorder=5)

        ov.text(x0, y+3.7, f"{cd:%b} {cd.day}", color=MUTED, fontsize=9, ha="center", va="center", zorder=5)

        if prev_click_x is not None and (x0-prev_click_x) > 22:
            mo = round((cd-prev_click_t).days/30.44)
            ov.text((x0+prev_click_x)/2, y-3.4, f"{mo} Months Later", color=MUTED, fontsize=9, style="italic", ha="center", va="center")

        prev_click_x, prev_click_t = seq[0], cd
def stat_card(ov, x, y, w, h, big, desc, fs_big=21, fs_desc=8.5, big_dx=11.5, desc_dx=20.5):
    rbox(ov, x, y, w, h, "white", 2.5, ec=CARD_EDGE, lw=1.2)
    ov.text(x+big_dx, y+h/2, big, color=ORANGE, fontsize=fs_big, fontweight="bold", ha="center", va="center")
    ov.text(x+desc_dx, y+h/2, desc, color=INK, fontsize=fs_desc, va="center", linespacing=1.3)
# BIGGEST SPENDER SLIDE
fig, ov = canvas()
header(ov, "Our Biggest Superfan Spender", f"Customer #{spender_id}")
t5 = fans.head(5).reset_index(drop=True)
rbox(ov, 97, 6.2, 1, 3, ORANGE, 0.4)
ov.text(99.5, 7.7, "Top 5 Spenders", color=MUTED, fontsize=10.5, va="center")
base, maxh = 25.5, 13
for i, (cid_, v) in enumerate(zip(t5["CustomerID"], t5["Revenue"])):
    bx = 99+i*12.6
    h_ = maxh*v/t5["Revenue"].max()
    ov.add_patch(FancyBboxPatch((bx, base-h_), 9.6, h_, boxstyle="round,pad=0,rounding_size=0.7", fc=ORANGE if i == 0 else PURPLE, ec="none"))
    ov.text(bx+4.8, base-h_-1.7, f"${v:,.0f}", color=INK, fontsize=10, fontweight="bold" if i == 0 else "normal", ha="center", va="center")
    ov.text(bx+4.8, base+2.4, f"#{int(cid_)}", color=INK if i == 0 else MUTED, fontsize=9, fontweight="bold" if i == 0 else "normal", ha="center", va="center")

ov.plot([98, 99+4*12.6+10.6], [base, base], color=INK, lw=1.1)
timeline_card(ov, spender_id, "BIGGEST SPENDER'S PURCHASE TIMELINE", "BIGGEST SPENDER", f"${top['Revenue']:,.0f} in {top['Orders']:.0f} orders", band=True)
stat_card(ov, TX, 72, TW, 12, f"{top['Revenue']/median_spend:.1f}x",
          f"The median purchaser's spend is $100. Customer {spender_id} is a true outlier.",
          fs_big=28, fs_desc=11.5, big_dx=16, desc_dx=32)

footnote(ov, "Notes: Biggest Spender = highest total spend. Each order is counted once (orders credited to two campaigns are not double counted).")
plt.savefig("q4_super_spender.png", dpi=200, facecolor="white")
plt.show()


In [ ]:
# GRAPH 4 - SLIDE 2: MOST LOYAL BUYER
fig, ov = canvas()
header(ov,"Our Biggest Superfan by Loyalty", f"Customer #{loyal_id}")
# PURCHASE TIMELINE
timeline_card(ov,loyal_id,"MOST LOYAL BUYER'S PURCHASE TIMELINE","MOST LOYAL BUYER",f"${loyal_rev:,.0f} over {loyal_days} purchase days")
# BOTTOM STATISTIC CARDS
cw = (TW - 3) / 2
# Percentage of purchasers who bought on multiple days
stat_card( ov, TX, 72, cw, 12, f"{n_multi / len(fans) * 100:.1f}%", f"{n_multi} of {len(fans):,} purchasers bought on more than one day.\nMost fans made just one purchase.",
    fs_big=24,fs_desc=10.5,big_dx=14,desc_dx=27)
# Number of purchasers who matched the loyalty leader
stat_card( ov, TX + cw + 3, 72, cw, 12, f"{n_max}", f"Purchasers bought on {loyal_days} separate days.\nCustomer {loyal_id} spent the most of them.", fs_big=24, fs_desc=10.5, big_dx=14, desc_dx=27)
footnote( ov, "Notes: Loyal Buyer = most separate calendar days with a purchase (ties broken by total spend). Each order is counted once (orders credited to two campaigns are not double counted).")
plt.savefig("q4_superfan_loyalty.png",dpi=200,facecolor="white")
plt.show()


In [ ]:
# Question 5: Where is our email campaigning most / least successful?
key = ["CustomerID", "ClickDate", "CampaignName"]

# one row per email click (a click that led to 2 orders appears twice in the raw data)
clicks = df.drop_duplicates(key).copy()
bought = df[df["Revenue"].notna()].drop_duplicates(key)[key].assign(Bought=1)
rev = df.groupby(key)["Revenue"].sum().rename("Rev").reset_index()
clicks = clicks.merge(bought, how="left", on=key).merge(rev, how="left", on=key)
clicks["Bought"] = clicks["Bought"].fillna(0)
clicks["Rev"] = clicks["Rev"].fillna(0)
clicks["State"] = clicks["CustomerID"].map(df.groupby("CustomerID")["State"].first())
total_clicks, total_rev = len(clicks), clicks["Rev"].sum()

# --- 1) Campaign scorecard ---
camp = clicks.groupby("CampaignName").agg(Clicks=("Bought", "size"), Buyers=("Bought", "sum"), Revenue=("Rev", "sum"))
camp["Conv%"] = camp["Buyers"] / camp["Clicks"] * 100                      # share of clicks that led to an order
camp["Rev/Click"] = camp["Revenue"] / camp["Clicks"]
camp["Rev/Order"] = camp["Revenue"] / df[df["Revenue"].notna()].groupby("CampaignName").size()
send_day = clicks.groupby("CampaignName")["ClickDate"].agg(lambda s: s.dt.date.mode()[0])   # day most clicks happened
camp["SendDay"] = pd.to_datetime(send_day).dt.strftime("%a %b %d, %Y")
camp = camp.sort_values("Rev/Click", ascending=False)
display(camp.round(2))

# --- 2) Best vs worst groups ---
groups = {"Holiday sends (C5, C6, C16)": ["Campaign 5", "Campaign 6", "Campaign 16"],"Jan-May sends (C7-C10)": ["Campaign 7", "Campaign 8", "Campaign 9", "Campaign 10"]}
print()
for name, c in groups.items():
    g = clicks[clicks["CampaignName"].isin(c)]
    print(f"{name}: {len(g)/total_clicks*100:.0f}% of clicks -> {g['Rev'].sum()/total_rev*100:.0f}% of revenue | "
          f"conversion {g['Bought'].mean()*100:.2f}% | ${g['Rev'].mean():.2f} per click")

# --- 3) Audience: past buyers and click fatigue ---
clicks = clicks.sort_values(["CustomerID", "ClickDate"])
clicks["PriorClicks"] = clicks.groupby("CustomerID").cumcount()
clicks["PastBuyer"] = (clicks.groupby("CustomerID")["Bought"].cumsum() - clicks["Bought"]) > 0
pb, non = clicks[clicks["PastBuyer"]], clicks[~clicks["PastBuyer"]]
print(f"\nPast buyers: {len(pb)/total_clicks*100:.1f}% of clicks, convert at {pb['Bought'].mean()*100:.1f}% vs {non['Bought'].mean()*100:.1f}% for everyone else")
fat = clicks[(clicks["PriorClicks"] >= 5) & (~clicks["PastBuyer"])]
print(f"Fatigued (5+ earlier clicks, never bought): {len(fat)/total_clicks*100:.1f}% of clicks, "
      f"convert at {fat['Bought'].mean()*100:.2f}%, bring in {fat['Rev'].sum()/total_rev*100:.1f}% of revenue")
display(clicks.groupby(clicks["PriorClicks"].clip(upper=5))["Bought"].agg(Clicks="size", Conv="mean").assign(Conv=lambda d: (d["Conv"]*100).round(2)))

# --- 4) How fast do people buy after clicking? ---
o = df[df["Revenue"].notna()].copy()
mins = (o["OrderDate"] - o["ClickDate"]).dt.total_seconds() / 60
print(f"\nOrders within 1 hour of the click: {(mins<=60).mean()*100:.0f}% | within 24 hours: {(mins<=1440).mean()*100:.0f}% | median: {mins.median():.0f} min")

# --- 5) State view (customers with a state listed, 300+ clicks) ---
st = clicks.groupby("State").agg(Clicks=("Bought", "size"), Conv=("Bought", "mean"), RevPerClick=("Rev", "mean"))
st = st[st["Clicks"] >= 300].sort_values("Conv", ascending=False)
st["Conv"] = (st["Conv"]*100).round(2); st["RevPerClick"] = st["RevPerClick"].round(2)
print(); display(st)

In [ ]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]
ORANGE, PURPLE = "#F56600", "#522D80"
INK, MUTED = "#1F1A2E", "#6B6480"
CARD, CARD_EDGE, GRID = "#F6F3FB", "#E3DDF0", "#E9E5F1"
HEAT = LinearSegmentedColormap.from_list("heat", ["#EFEBF6", "#F9D3B4", ORANGE])   # low -> high

# --- table + numbers (uses camp, clicks, pb, non, fat, mins, send_day from the analysis cell) ---
tab = camp.copy()
tab["Date"] = pd.to_datetime(send_day)
tab = tab.sort_values("Date")                                   # chronological, so seasonality shows
elig = camp[camp["Clicks"] >= 1000]                             # ignore tiny sends when picking best / worst
best, worst = list(elig.nlargest(3, "Rev/Click").index), list(elig.nsmallest(4, "Rev/Click").index)
cb, cw = clicks[clicks["CampaignName"].isin(best)], clicks[clicks["CampaignName"].isin(worst)]
ratio = cb["Rev"].mean() / cw["Rev"].mean()
best_rev_share, best_click_share = cb["Rev"].sum() / total_rev * 100, len(cb) / total_clicks * 100

W, H = 165, 93
fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1]); ov.set_xlim(0, W); ov.set_ylim(H, 0); ov.axis("off")
def rbox(x, y, w, h, fc, r, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x + r, y + r), w - 2*r, h - 2*r,
                 boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

# ---- header ----
L = 6
rbox(L, 7.2, 1.2, 3.2, ORANGE, 0.5)
ov.text(L + 3, 8.8, "CLEMSON ATHLETICS  ·  FAN COMMERCE", color=MUTED, fontsize=9.5, va="center")
ov.text(L, 17.5, "Holiday Email Campaigns Earn", color=INK, fontsize=26, fontweight="bold", va="center")
ov.text(L, 25, f"{ratio:.1f}x More per Click than Offseason", color=ORANGE, fontsize=26, fontweight="bold", va="center")

# ---- heatmap table ----
RH, Y0 = 3.15, 36.4
heat_cols = [("Conv%", "CONV.", 55, 13, "{:.1f}%"), ("Rev/Click", "REV / CLICK", 69, 13, "{:.2f}"), ("Rev/Order", "REV / ORDER", 83, 13, "{:.0f}")]
for hx, lab in [(8, "CAMPAIGN"), (27, "SEND DAY")]:
    ov.text(hx, 33.7, lab, color=MUTED, fontsize=7.5, va="center")
ov.text(52, 33.7, "CLICKS", color=MUTED, fontsize=7.5, ha="right", va="center")
ov.text(106.5, 33.7, "REVENUE", color=MUTED, fontsize=7.5, ha="right", va="center")
for col, lab, cx, cw_, fmt in heat_cols:
    ov.text(cx + cw_ / 2, 33.7, lab, color=MUTED, fontsize=7.5, ha="center", va="center")
ov.plot([5, 106.5], [35.2, 35.2], color=CARD_EDGE, lw=1.1)

for i, (name, r) in enumerate(tab.iterrows()):
    y = Y0 + i * RH; h = RH - 0.4
    small = r["Clicks"] < 400
    if name in best or name in worst:
        ov.add_patch(FancyBboxPatch((5, y), 1.4, h, boxstyle="round,pad=0,rounding_size=0.5", fc=ORANGE if name in best else PURPLE, ec="none"))
    ov.text(8, y + h / 2, name + ("*" if small else ""), color=MUTED if small else INK, fontsize=8.5, fontweight="normal" if small else "bold", va="center")
    d = r["Date"]
    ov.text(27, y + h / 2, f"{d:%a} {d:%b} {d.day}, {d:%Y}", color=MUTED, fontsize=8.2, va="center")
    ov.text(52, y + h / 2, f"{r['Clicks']:,.0f}", color=MUTED, fontsize=8.5, ha="right", va="center")
    for col, lab, cx, cw_, fmt in heat_cols:
        lo, hi = tab[col].min(), tab[col].max(); t = (r[col] - lo) / (hi - lo)
        ov.add_patch(FancyBboxPatch((cx, y), cw_, h, boxstyle="round,pad=0,rounding_size=0.5", fc=HEAT(t), ec="none"))
        txt = ("$" if col != "Conv%" else "") + fmt.format(r[col])
        ov.text(cx + cw_ / 2, y + h / 2, txt, color="white" if t > 0.72 else INK, fontsize=8.5, fontweight="bold" if t > 0.72 else "normal", ha="center", va="center")
    ov.text(106.5, y + h / 2, f"${r['Revenue']:,.0f}", color=MUTED if small else INK, fontsize=8.5, fontweight="normal" if small else "bold", ha="right", va="center")

# ---- top-right: how to read ----
KX = 111
rbox(KX, 6.2, 1.0, 3.0, ORANGE, 0.4)
ov.text(KX + 2.5, 7.7, "HOW TO READ", color=MUTED, fontsize=9, va="center")
for k in range(20):
    ov.add_patch(Rectangle((KX + k * 1.2, 11.5), 1.25, 2.4, fc=HEAT(k / 19), ec="none"))
ov.text(KX + 25.2, 12.7, "Lower  →  Higher (per column)", color=MUTED, fontsize=8, va="center")
for j, (c, lab) in enumerate([(ORANGE, "Top 3 per click: Black Friday & Cyber Monday sends"),(PURPLE, "Bottom 4 per click: Offseason sends (Jan–May)")]):
    ov.add_patch(FancyBboxPatch((KX, 17.0 + j * 4.2), 1.4, 3.0, boxstyle="round,pad=0,rounding_size=0.5", fc=c, ec="none"))
    ov.text(KX + 3.2, 18.5 + j * 4.2, lab, color=INK, fontsize=8, va="center")

# ---- right: how to make it better ----
rbox(KX, 27.6, 1.0, 3.0, ORANGE, 0.4)
ov.text(KX + 2.5, 29.1, "HOW TO MAKE IT BETTER", color=MUTED, fontsize=9, va="center")
cards = [("Send into shopping moments", f"{best_rev_share:.0f}%", f"of revenue came from 3 holiday\nsends (just {best_click_share:.0f}% of clicks)"),
         ("Win back past buyers", f"{pb['Bought'].mean()*100:.1f}%", f"conversion for past buyers vs\n{non['Bought'].mean()*100:.1f}% for others; {len(pb)/total_clicks*100:.1f}% of clicks"),
         ("Follow up within hours", f"{(mins <= 60).mean()*100:.0f}%", f"of orders land within an hour of\nthe click ({(mins <= 1440).mean()*100:.0f}% within 24 hrs)")]
CH_, GAP = 16.6, 2.0
for j, (title, stat, desc) in enumerate(cards):
    y = 32.4 + j * (CH_ + GAP)
    rbox(KX, y, 49, CH_, CARD, 2.4, ec=CARD_EDGE, lw=1.1)
    ov.add_patch(Circle((KX + 4.4, y + 4.6), 2.0, fc=ORANGE, ec="none"))
    ov.text(KX + 4.4, y + 4.6, str(j + 1), color="white", fontsize=9, fontweight="bold", ha="center", va="center")
    ov.text(KX + 8.4, y + 4.6, title, color=INK, fontsize=10.5, fontweight="bold", va="center")
    ov.text(KX + 3.6, y + 11.6, stat, color=ORANGE, fontsize=19, fontweight="bold", va="center")
    ov.text(KX + 19.5, y + 11.4, desc, color=MUTED, fontsize=8.5, va="center", linespacing=1.3)

# ---- footnote ----
ov.text(L, H - 4.0, "Conversion = share of email clicks that led to an order (the data covers clicks, not sends or opens). "
        "Send day = the day with the most clicks for that campaign.", color=MUTED, fontsize=8, va="center")
ov.text(L, H - 1.7, "* Under 400 clicks, so not ranked.  Past buyer = a customer with an earlier order.", color=MUTED, fontsize=8, va="center")
plt.savefig("q5_campaigns.png", dpi=200, facecolor="white")
plt.show()

In [ ]:
# QUESTION 5 - SLIDE 1: CAMPAIGN PERFORMANCE
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"]

ORANGE, PURPLE = "#F56600", "#522D80"
INK, MUTED = "#1F1A2E", "#6B6480"
CARD, CARD_EDGE = "#F6F3FB", "#E3DDF0"
HEAT = LinearSegmentedColormap.from_list("heat", ["#EFEBF6", "#F9D3B4", ORANGE])

# Prepare campaign table
tab = camp.copy()
tab["Date"] = pd.to_datetime(send_day)
tab = tab.sort_values("Date")

elig = camp[camp["Clicks"] >= 1000]
best = list(elig.nlargest(3, "Rev/Click").index)
worst = list(elig.nsmallest(4, "Rev/Click").index)

cb = clicks[clicks["CampaignName"].isin(best)]
cw = clicks[clicks["CampaignName"].isin(worst)]
ratio = cb["Rev"].mean() / cw["Rev"].mean()

# Canvas
W, H = 165, 93
fig = plt.figure(figsize=(13.33, 13.33 * H / W), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1])
ov.set_xlim(0, W)
ov.set_ylim(H, 0)
ov.axis("off")

def rbox(x, y, w, h, fc, r=1, ec="none", lw=1):
    ov.add_patch(FancyBboxPatch((x+r, y+r), w-2*r, h-2*r,
        boxstyle=f"round,pad={r},rounding_size={r}", fc=fc, ec=ec, lw=lw))

# Header
rbox(6, 7.2, 1.2, 3.2, ORANGE, 0.5)
ov.text(9, 8.8, "Clemson Athletics    Fan Commerce", color=MUTED, fontsize=11, va="center")
ov.text(6, 17.5, "Holiday Email Campaigns Earn", color=INK, fontsize=26, fontweight="bold", va="center")
ov.text(6, 25, f"{ratio:.1f}x More per Click than Offseason", color=ORANGE, fontsize=26, fontweight="bold", va="center")

# Heatmap table
Y0, RH = 39, 2.95
heat_cols = [("Conv%", 57, 13, "{:.1f}%"), ("Rev/Click", 72, 14, "{:.2f}"), ("Rev/Order", 88, 14, "{:.0f}")]

ov.text(8, 36.5, "CAMPAIGN", color=MUTED, fontsize=9, fontweight="bold")
ov.text(29, 36.5, "SEND DAY", color=MUTED, fontsize=9, fontweight="bold")
ov.text(54, 36.5, "CLICKS", color=MUTED, fontsize=9, fontweight="bold", ha="right")
ov.text(109, 36.5, "REVENUE", color=MUTED, fontsize=9, fontweight="bold", ha="right")
for col, x, width, fmt in heat_cols:
    ov.text(x+width/2, 36.5, col.upper(), color=MUTED, fontsize=8.5, fontweight="bold", ha="center")

ov.plot([5, 110], [37.5, 37.5], color=CARD_EDGE, lw=1.1)

for i, (name, r) in enumerate(tab.iterrows()):
    y = Y0 + i*RH
    h = RH-0.3
    small = r["Clicks"] < 400

    if name in best or name in worst:
        rbox(5, y, 1.3, h, ORANGE if name in best else PURPLE, 0.4)

    ov.text(8, y+h/2, name+("*" if small else ""), color=MUTED if small else INK,
            fontsize=9.5, fontweight="bold" if not small else "normal", va="center")
    d = r["Date"]
    ov.text(29, y+h/2, f"{d:%a} {d:%b} {d.day}, {d:%Y}", color=MUTED, fontsize=8.8, va="center")
    ov.text(54, y+h/2, f"{r['Clicks']:,.0f}", color=INK, fontsize=9, ha="right", va="center")

    for col, x, width, fmt in heat_cols:
        lo, hi = tab[col].min(), tab[col].max()
        t = (r[col]-lo)/(hi-lo) if hi != lo else 0.5
        ov.add_patch(FancyBboxPatch((x, y), width, h,
            boxstyle="round,pad=0,rounding_size=0.4", fc=HEAT(t), ec="none"))
        txt = ("$" if col != "Conv%" else "") + fmt.format(r[col])
        ov.text(x+width/2, y+h/2, txt, color="white" if t > 0.72 else INK,
                fontsize=9.5, fontweight="bold", ha="center", va="center")

    ov.text(109, y+h/2, f"${r['Revenue']:,.0f}", color=INK if not small else MUTED,
            fontsize=9, fontweight="bold" if not small else "normal", ha="right", va="center")

# How to Read panel
KX = 116
rbox(KX, 6.2, 1, 3, ORANGE, 0.4)
ov.text(KX+3, 7.7, "HOW TO READ", color=INK, fontsize=11, fontweight="bold", va="center")

ov.text(KX, 12, "Heatmap Colors", color=INK, fontsize=11, fontweight="bold")
for k in range(25):
    ov.add_patch(Rectangle((KX+k*1.15, 14), 1.2, 2.8, fc=HEAT(k/24), ec="none"))

ov.text(KX, 19.5, "Lower", color=MUTED, fontsize=9.5)
ov.text(145, 19.5, "Higher", color=MUTED, fontsize=9.5, ha="right")
ov.text(KX, 25, "Compared within each metric column.", color=MUTED, fontsize=9)

# Campaign ranking legend
rbox(KX, 30, 45, 17, CARD, 2, ec=CARD_EDGE)
ov.add_patch(Rectangle((KX+3, 34), 2, 2.5, fc=ORANGE, ec="none"))
ov.text(KX+7, 35.2, "Top 3 per click", color=INK, fontsize=11, fontweight="bold", va="center")
ov.text(KX+7, 39, "Black Friday & Cyber Monday", color=MUTED, fontsize=9.5, va="center")

ov.add_patch(Rectangle((KX+3, 42), 2, 2.5, fc=PURPLE, ec="none"))
ov.text(KX+7, 43.2, "Bottom 4 per click", color=INK, fontsize=11, fontweight="bold", va="center")
ov.text(KX+7, 46, "Offseason sends (Jan-May)", color=MUTED, fontsize=9.5, va="center")

# Footnotes
ov.text(6, 90, "Conversion = share of email clicks that led to an order. Data covers clicks, not sends or opens.",
        color=MUTED, fontsize=10.5, va="center")
ov.text(6, 92, "Send day = day with the most clicks for that campaign. * Under 400 clicks, so not ranked.",
        color=MUTED, fontsize=10.5, va="center")

plt.savefig("q5_campaign_performance.png", dpi=200, facecolor="white")
plt.show()


In [ ]:
# Q5 Conclusion Slide
fig, ov = plt.subplots(figsize=(13.33, 7.5))
fig.set_facecolor("white")
ov.remove()
fig = plt.figure(figsize=(13.33, 7.5), facecolor="white")
ov = fig.add_axes([0, 0, 1, 1])
ov.set_xlim(0, 165); ov.set_ylim(93, 0); ov.axis("off")

# Header
rbox(6, 7.2, 1.2, 3.2, ORANGE, 0.5)
ov.text(9, 8.8, "Clemson Athletics    Fan Commerce", color=MUTED, fontsize=10, va="center")
ov.text(6, 17.5, "Conclusion: Turning Email Clicks Into Revenue", color=INK, fontsize=26, fontweight="bold", va="center")
ov.text(6, 25, "Three Opportunities to Improve Campaign Performance", color=ORANGE, fontsize=22, fontweight="bold", va="center")

# Card data
cards = [
    ("1", "Send Into Shopping Moments", f"{best_rev_share:.0f}%",
     f"of revenue came from the 3 top-performing\nholiday sends, despite generating only\n{best_click_share:.0f}% of clicks.",
     "Prioritize Black Friday, Cyber Monday,\nand other high-intent shopping periods."),

    ("2", "Win Back Past Buyers", f"{pb['Bought'].mean()*100:.1f}%",
     f"conversion for past buyers compared with\n{non['Bought'].mean()*100:.1f}% for customers\nwithout a previous purchase.",
     f"Past buyers accounted for\n{len(pb)/total_clicks*100:.1f}% of all clicks."),

    ("3", "Follow Up Within Hours", f"{(mins <= 60).mean()*100:.0f}%",
     f"of orders occurred within one hour of a\nclick. {(mins <= 1440).mean()*100:.0f}% occurred within 24 hours.",
     "Keep follow-up emails and relevant\noffers timely.")
]

# Three cards
xs = [7, 60, 113]
for x, (num, title, stat, desc, action) in zip(xs, cards):
    rbox(x, 34, 45, 37, CARD, 2.5, ec=CARD_EDGE, lw=1.2)

    ov.add_patch(Circle((x + 5, 39), 2.7, fc=ORANGE, ec="none"))
    ov.text(x + 5, 39, num, color="white", fontsize=11, fontweight="bold", ha="center", va="center")

    ov.text(x + 9, 39, title, color=INK, fontsize=9.5, fontweight="bold", va="center")
    ov.text(x + 4, 48, stat, color=ORANGE, fontsize=24, fontweight="bold", va="center")
    ov.text(x + 4, 55, desc, color=INK, fontsize=8.5, va="center", linespacing=1.5)
    ov.text(x + 4, 66, action, color=MUTED, fontsize=8, va="center", linespacing=1.4)

# Big takeaway
rbox(7, 76, 151, 10, "white", 2.2, ec=CARD_EDGE, lw=1.2)
ov.text(10, 79.5, "THE BIG TAKEAWAY", color=ORANGE, fontsize=10.5, fontweight="bold", va="center")
ov.text(10, 83.5, "Focus on high-intent seasons, prioritize existing buyers, and act quickly after engagement.",
        color=INK, fontsize=11, fontweight="bold", va="center")

# Footnote
ov.text(7, 91, "Notes: Recommendations based on campaign revenue per click, customer purchase history, and click-to-order timing.",
        color=MUTED, fontsize=10, va="center")

plt.savefig("q5_conclusion.png", dpi=200, facecolor="white")
plt.show()